# Train + evaluate ONE variant (Kaggle)

**Before running:** Settings -> Accelerator **GPU T4**, Internet **On**.
Add Input: `ceymo-official` dataset, `experiments-code` dataset, the baseline training notebook output
(`notebook94749b7315`), and the output of every earlier session of this notebook (so unfinished runs resume).

Set `VARIANT` / `SEED` in cell 1, then **Save Version -> Save & Run All (Commit)**. One variant per session.
Safe to re-run: finished runs are skipped, interrupted runs resume from `last.pt`, existing results are not recomputed.


In [ ]:
# 1) WHAT TO RUN -- change only these two lines
VARIANT = "full"   # baseline | baseline_p256 | msem_only | pr_only | full
SEED    = 0        # 0, 1, 2

TAG = "flipswap_640"; FLIP = "swap"; IMGSZ = 640
RUNS, RESULTS = f"runs_{TAG}", f"results_{TAG}"
WORK = "/kaggle/working/experiments"


In [ ]:
# 2) Setup: code, fixed evaluate.py, earlier runs/results, ultralytics, dataset
import glob, os, shutil, subprocess
from pathlib import Path

def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

# code: any attached 'experiments' folder that has train_variant.py (notebook outputs first)
srcs = sorted(glob.glob("/kaggle/input/**/train_variant.py", recursive=True),
              key=lambda p: ("/notebooks/" not in p, len(p)))
assert srcs, "No experiments code found in inputs - attach experiments-code or the training notebook output"
code_src = Path(srcs[0]).parent
print("code from:", code_src)
skip = shutil.ignore_patterns("runs*", "results*", "datasets", "__pycache__", "*.pt")
shutil.copytree(code_src, WORK, ignore=skip, dirs_exist_ok=True)

# evaluate.py with the FPS fix
fixed = [p for p in glob.glob("/kaggle/input/**/evaluate.py", recursive=True) if "Load a fresh copy" in open(p).read()]
assert fixed, "Fixed evaluate.py not found - upload experiments_kaggle.zip (Nov 15) as the code dataset, not experiments_for_colab.zip"
shutil.copy(fixed[0], f"{WORK}/evaluate.py"); print("evaluate.py from:", fixed[0])

# earlier runs: copy each run folder; prefer a finished copy (DONE), else the one with the newest last.pt
def progress(d):
    d = Path(d)
    if (d / "DONE").exists(): return float("inf")
    lp = d / "weights" / "last.pt"
    return lp.stat().st_mtime if lp.exists() else -1
for d in glob.glob(f"/kaggle/input/**/{RUNS}/*_s*", recursive=True):
    dst = Path(WORK, RUNS, Path(d).name)
    if not dst.exists() or progress(d) > progress(dst):
        shutil.rmtree(dst, ignore_errors=True); shutil.copytree(d, dst); print("restored run:", d)
os.makedirs(f"{WORK}/{RESULTS}", exist_ok=True)
for f in glob.glob(f"/kaggle/input/**/{RESULTS}/*.json", recursive=True):
    if not Path(WORK, RESULTS, Path(f).name).exists():
        shutil.copy(f, f"{WORK}/{RESULTS}/"); print("restored result:", f)

os.chdir(WORK)
sh("pip install -q ultralytics==8.4.90 pyyaml opencv-python")
sh("python patch_ultralytics.py")
sh("python make_variant_yamls.py")

# CeyMo -> YOLO-seg (deterministic; identical to split_stats_seed0.json)
os.makedirs("/kaggle/working/ceymo", exist_ok=True)
# find the CeyMo train/test folders in ANY attached dataset (works on every Kaggle account)
for s in ("train", "test"):
    hits = sorted({str(Path(p).parent.parent) for p in glob.glob("/kaggle/input/**/polygon_annotations/*.json", recursive=True)
                   if Path(p).parent.parent.name == s and Path(p).parent.parent.joinpath("images").is_dir()}, key=len)
    assert hits, f"CeyMo '{s}' folder (with images/ and polygon_annotations/) not found under /kaggle/input"
    print(f"CeyMo {s}:", hits[0])
    sh(f"ln -sfn '{hits[0]}' /kaggle/working/ceymo/{s}")
DATA = f"{WORK}/datasets/ceymo_official/data.yaml"
if not os.path.exists(DATA):
    sh("python convert_ceymo.py --ceymo /kaggle/working/ceymo --out datasets/ceymo_official --symlink")
print(open("datasets/ceymo_official/split_stats.json").read()[:400])


In [ ]:
# 3) Use exactly the baseline's batch size (fair comparison)
import yaml
base_args = glob.glob(f"/kaggle/input/**/{RUNS}/baseline_s0/args.yaml", recursive=True) + glob.glob(f"{WORK}/{RUNS}/baseline_s0/args.yaml")
# baseline_s0 trained with batch 8; fall back to that if its notebook output is not attached (other account)
BATCH = yaml.safe_load(open(base_args[0]))["batch"] if base_args else 8
WORKERS = 2
print("BATCH =", BATCH, "| WORKERS =", WORKERS)


In [ ]:
# 4) Train (skips if finished, resumes from last.pt if interrupted)
sh(f"python train_variant.py --variant {VARIANT} --data {DATA} --seed {SEED} --batch {BATCH} "
   f"--device 0 --workers {WORKERS} --project {WORK}/{RUNS} --imgsz {IMGSZ} --flip {FLIP}")
assert Path(WORK, RUNS, f"{VARIANT}_s{SEED}", "DONE").exists(), "training did not finish"


In [ ]:
# 5) Evaluate on val and test (FPS on test, seed 0 only) - same protocol as run_all.sh
w = f"{RUNS}/{VARIANT}_s{SEED}/weights/best.pt"
for split in ("val", "test"):
    out = Path(RESULTS, f"{VARIANT}_s{SEED}_{split}.json")
    if out.exists():
        print("exists, skipping:", out); continue
    fps = "--fps" if (split == "test" and SEED == 0) else ""
    sh(f"python evaluate.py --weights {w} --data {DATA} --split {split} --imgsz {IMGSZ} "
       f"--variant {VARIANT}_s{SEED} --out {RESULTS} --device 0 --workers {WORKERS} {fps}")


In [ ]:
# 6) Summary + one zip to download (results + this run's weights/curves)
import json
for f in sorted(Path(RESULTS).glob("*_test.json")):
    r = json.load(open(f)); m = r["mask"]
    print(f"{r['variant']:<18} test  mAP50 {m['mAP50']*100:5.1f}  mAP {m['mAP50_95']*100:5.1f}  "
          f"mIoU {r['mIoU']*100:5.1f}  BIoU3 {r['BIoU_3px']*100:5.1f}  "
          f"FPS {((r.get('speed') or {}).get('fps') or 0):5.1f}  params {r['params_M']:.2f}M")
run = f"{RUNS}/{VARIANT}_s{SEED}"
sh(f"cd {WORK} && zip -qr /kaggle/working/{VARIANT}_s{SEED}_{TAG}.zip {RESULTS} "
   f"{run}/weights/best.pt {run}/results.csv {run}/args.yaml {run}/recipe.json")
sh(f"ls -lh /kaggle/working/{VARIANT}_s{SEED}_{TAG}.zip")
